# Week 9: Encoder‑Decoder Generation (T5)

Welcome to Week 9! In this module, we will explore **sequence-to-sequence (seq2seq)** encoder-decoder models using the **text-to-text** framing popularized by Google's T5 (Text-to-Text Transfer Transformer). Unlike decoder-only architectures, the encoder-decoder design separates understanding (encoding the source) from generation (decoding the target).

## 🎯 Course Goals & Notebook Structure

This notebook is structured into three progressive parts to build both high-level intuition and concrete engineering skills:

### [Part A — What changes vs GPT?](#tiny-t5-intro)
- **Concept**: We contrast the **decoder-only (GPT)** architecture with the **encoder-decoder (T5)** design.
- **Theory**: Learn how the fully bidirectional **Encoder** builds context-rich representations of source sentences, how the **Decoder** leverages causal masking, and how the **Cross-Attention** bridge connects them to perform sequence alignment.

### [Part B — Tiny Encoder‑Decoder Transformer in Pure PyTorch](#tiny-t5-architecture)
- **Scratch Build**: We write a modular, consolidated Tiny T5 model from scratch using clean PyTorch classes.
- **Key Mechanics**: Experiment directly with causal masking, absolute positional embeddings, and weight-tied embeddings (`lm_head.weight = embedding.weight`).
- **Objective**: Train the model on a toy **sequence-reversal task** to observe real-world optimization challenges, like how under-trained models cheat using a "copying shortcut" and how they undergo a sudden phase change to achieve perfect convergence at 1000 steps.

### [Part C — Hugging Face T5 Translation (CPU‑friendly)](#5635c02e)
- **Application**: Move from scratch implementations to pre-trained architectures using `t5-small`.
- **Fine-Tuning**: Run a CPU-friendly fine-tuning cycle using Hugging Face's `Trainer` and the `opus100` English-to-German translation dataset.
- **Optimization**: Learn how to use small slices, truncation, and gradient accumulation steps to fine-tune a seq2seq translation model on standard laptop hardware.

---

## 📝 Important Notes
- **Hugging Face Download**: The first run in Part C will download the `opus100` dataset and `t5-small` tokenizer/model configurations. Ensure you have an active internet connection.
- **Hardware Friendly**: All configurations (dataset slices, training epochs, and batch sizes) are scaled down to be extremely friendly for standard CPU or local MPS (Mac metal acceleration) runtimes.

In [1]:
# Setup
from utils import clean_memory, get_device
import numpy as np
import torch


seed = 204
torch.manual_seed(seed)

device = get_device()
device

device(type='mps')

## Part A — What changes vs GPT?

To understand the **Encoder‑Decoder (T5)** architecture, it is helpful to compare it directly to a **Decoder‑Only (GPT)** model:

### 1. Decoder‑Only Models (e.g., GPT)
- **Structure**: Consists of a single stack of Transformer blocks.
- **Attention**: Uses **causal self‑attention**, where each token is masked so it can only look at previous tokens (and itself). It is designed to predict the *next* token in a single, continuous stream.
- **Ideal For**: Open-ended text generation, creative writing, and general conversational autocomplete.

### 2. Encoder‑Decoder Models (e.g., T5)
- **Structure**: Consists of two distinct, coupled stacks—an **Encoder** and a **Decoder**.
- **The Encoder (Bidirectional)**:
  - Processes the entire source input (e.g., *"I really enjoyed this movie"*).
  - Uses **bidirectional (unmasked) self‑attention**, meaning every source token can attend to every other source token, regardless of position. This allows the encoder to build a rich, context-aware representation of the complete input sequence.
- **The Decoder (Causal + Cross-Attention)**:
  - Generates the target sequence (e.g., German translation *"Ich habe diesen Film wirklich genossen"*) token-by-token.
  - Uses **Causal Self-Attention** so that generated target tokens can only look back at previously generated target tokens.
  - Uses **Cross-Attention** (or Encoder-Decoder Attention) to project queries ($Q$) from the decoder states into key-value pairs ($K, V$) originating from the encoder's final representations. This is the "translation bridge" where the decoder decides which source words to focus on at any given moment.
- **Training Objective**: Trained using **Teacher Forcing** with a shifted-right label mechanism. The decoder is fed the actual target sentence shifted by one position, allowing the entire target sequence to be processed in parallel during training rather than sequentially.

## Part B — Tiny encoder-decoder Transformer in pure PyTorch

This section builds a toy T5-style encoder-decoder Transformer from scratch to make the seq2seq mechanics concrete. Let's walk through the core engineering components:

### 1. Causal and Padding Masking
- **Source Mask**: A simple boolean mask of shape `(batch, 1, 1, src_len)` that prevents the encoder from attending to empty `<PAD>` tokens at the end of the input.
- **Target Mask**: A combined mask of shape `(batch, 1, tgt_len, tgt_len)` that applies a logical `AND` between the padding mask and a lower-triangular causal mask (`torch.tril`). This prevents the decoder from "looking into the future" while ignoring pad tokens.

### 2. The Cross-Attention Mechanism
- In our consolidated `MultiHeadAttention` module, the **cross-attention** is executed by passing the decoder's hidden states as the `query` ($Q$), and the encoder's outputs as the `context` to generate keys and values ($K, V$):
  $$\text{CrossAttention}(X_{\text{dec}}, H_{\text{enc}}) = \text{Softmax}\left(\frac{Q_{\text{dec}} K_{\text{enc}}^T}{\sqrt{d_k}}\right) V_{\text{enc}}$$
- This aligns target-side words directly with context-side words.

### 3. Teacher Forcing & Shift-Right
- During training, the decoder reads the target labels. To prevent the model from simply copying the label at index $t$ to predict index $t$, the labels are shifted to the right by prepending a `<BOS>` (Beginning of Sequence, `1`) token:
  $$\text{Target}: [\text{Word}_1, \text{Word}_2, \text{Word}_3, \text{EOS}] \rightarrow \text{Decoder Input}: [\text{BOS}, \text{Word}_1, \text{Word}_2, \text{Word}_3]$$
- This forces the decoder to predict Word $t$ using only context and words up to step $t-1$.

### 4. Weight Tying Optimization
- We tie the weights of the input `embedding` layer and the output projection `lm_head` (`self.lm_head.weight = self.embedding.weight`).
- This reduces model parameters by nearly half (since vocab projection matrices are massive) and acts as a powerful regularizer, forcing the representation spaces of input and output tokens to remain perfectly aligned.

### 5. Why the "Copying Shortcut" Happens initially
- When training a seq2seq model for a very small number of steps (e.g., 80), the model often learns a "shortcut" where the causal self-attention simply copies the input token it just saw (lagging by one step) because this is a very simple linear mapping.
- Increasing training steps (e.g., to 1000) forces the cross-attention alignment to converge, breaking the copying shortcut and teaching the model to output the true shifted target sequences!

### 6. Production Optimization: Key‑Value (KV) Caching & Causal Mask Offsets
- **Encoder KV Cache (Cross-Attention)**: Since the encoder's representations are static, we project them once to get Key and Value, and cache them (`past_key_value`). In subsequent decoding steps, we bypass `k_proj` and `v_proj` entirely, reducing cross-attention projection overhead to zero!
- **Decoder KV Cache (Self-Attention)**: Instead of re-evaluating the entire sequence, we pass only the single new token at step $t$ (query length = 1) and concatenate its new $K/V$ to the cached history. This drops generation complexity per step from $O(t^2)$ to $O(t)$.
- **Causal Mask Offset**: When decoding a chunk of new tokens $C$ at an index offset $S$ (where the cache contains $S$ tokens), we cannot use a simple `tril` mask. Instead, we use position comparisons to build a causal mask of shape `(C, S + C)`:
  $$\text{mask}[i, j] = \text{True} \quad \text{if} \quad j \le i + S \quad \text{else} \quad \text{False}$$
- This makes the custom `MultiHeadAttention` fully self-contained and ready for high-throughput, low-latency production inference!

In [2]:
# Import our clean implementation from utils.py
from utils import TinyT5, train_tiny_T5, evaluate_tiny_T5, make_reverse_batch

In [5]:
# Tiny synthetic seq2seq task: map source tokens to the reversed target sequence.
PAD, BOS, EOS = 0, 1, 2
vocab_size = 32
src_len = 8
tgt_len = src_len + 1  # reversed source plus EOS

# Clean memory before training as requested
if "tiny_t5" in globals():
    clean_memory(vars_to_delete=["tiny_t5"], scope=globals(), verbose=True)
else:
    clean_memory(verbose=True)

# Initialize TinyT5 directly using the clean class from utils.py
tiny_t5 = TinyT5(
    vocab_size=vocab_size,
    d_model=128,
    num_heads=4,
    mlp_hidden_dim=128,
    num_layers=2,
    pad_id=PAD,
    bos_id=BOS,
    max_src_len=src_len,
    max_tgt_len=tgt_len,
).to(device)

# Train model using the library training loop and save to models/tiny_t5
tiny_t5 = train_tiny_T5(
    model=tiny_t5,
    device=device,
    epochs=1000,
    lr=3e-4,
    batch_size=64,
    vocab_size=vocab_size,
    src_len=src_len,
    EOS=EOS,
    pad_id=PAD,
    save_model_path="models/tiny_t5/best_model.pt",
)

# Evaluate and print prediction results using evaluate_tiny_T5 directly
src_list, target_list, pred_list = evaluate_tiny_T5(
    model=tiny_t5,
    batch_size=1,
    vocab_size=vocab_size,
    src_len=src_len,
    EOS=EOS,
    device=device,
)

print("source: ", src_list)
print("target: ", target_list)
print("pred:   ", pred_list)


Memory cleaned.
Saved best model state to models/tiny_t5/best_model.pt
epoch 1 | loss 80.8641
Saved best model state to models/tiny_t5/best_model.pt
Saved best model state to models/tiny_t5/best_model.pt
Saved best model state to models/tiny_t5/best_model.pt
Saved best model state to models/tiny_t5/best_model.pt
Saved best model state to models/tiny_t5/best_model.pt
Saved best model state to models/tiny_t5/best_model.pt
Saved best model state to models/tiny_t5/best_model.pt
Saved best model state to models/tiny_t5/best_model.pt
Saved best model state to models/tiny_t5/best_model.pt
Saved best model state to models/tiny_t5/best_model.pt
Saved best model state to models/tiny_t5/best_model.pt
Saved best model state to models/tiny_t5/best_model.pt
Saved best model state to models/tiny_t5/best_model.pt
Saved best model state to models/tiny_t5/best_model.pt
Saved best model state to models/tiny_t5/best_model.pt
Saved best model state to models/tiny_t5/best_model.pt
Saved best model state to 

## Part C — Hugging Face T5 translation (CPU‑friendly)

Since `opus100` (en-de) is occasionally unavailable or experiences server issues on Hugging Face, we use the highly available, robust, and lightweight `opus_books` (`de-en`) dataset for German-to-English translation. This fits perfectly with `t5-small`'s pre-training (which natively supports WMT German-English translation).

To keep this training cycle extremely realistic and fast on standard CPU or local MPS (Mac metal acceleration) runtimes, we:
- Partition the single-split `opus_books` dataset into 90% train and 10% validation sets.
- Take small slices (e.g. 4000 training, 500 validation examples).
- Cap maximum source and target token lengths to 128.
- Train for a compact number of steps using gradient accumulation.

In [8]:
import os
from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
)

hf_model_name = "t5-small"

# CPU-friendly sizes
n_train = 4000
n_val = 500

max_source_len = 128
max_target_len = 128

# Load opus_books for German-English translation (de-en) as a robust, lightweight alternative
raw = load_dataset("opus_books", "de-en")

# Since opus_books only contains a 'train' split, we partition it into 90% train and 10% validation
split_ds = raw["train"].train_test_split(test_size=0.1, seed=seed)
raw = DatasetDict({
    "train": split_ds["train"],
    "validation": split_ds["test"]
})
raw

DatasetDict({
    train: Dataset({
        features: ['id', 'translation'],
        num_rows: 46320
    })
    validation: Dataset({
        features: ['id', 'translation'],
        num_rows: 5147
    })
})

In [9]:
tokenizer = AutoTokenizer.from_pretrained(hf_model_name)

# Set prefix for German to English translation
prefix = "translate German to English: "

def preprocess(batch):
    src_texts = [prefix + ex["de"] for ex in batch["translation"]]
    tgt_texts = [ex["en"] for ex in batch["translation"]]

    # Using the modern text_target argument to avoid using the deprecated as_target_tokenizer() call
    model_inputs = tokenizer(
        src_texts,
        max_length=max_source_len,
        truncation=True,
        text_target=tgt_texts,
    )
    return model_inputs

tok = raw.map(preprocess, batched=True, remove_columns=raw["train"].column_names)

train_ds = tok["train"].shuffle(seed=seed).select(range(min(n_train, len(tok["train"])) ))
val_ds = tok["validation"].shuffle(seed=seed).select(range(min(n_val, len(tok["validation"])) ))
train_ds, val_ds

Map:   0%|          | 0/5147 [00:00<?, ? examples/s]

(Dataset({
     features: ['input_ids', 'attention_mask', 'labels'],
     num_rows: 4000
 }),
 Dataset({
     features: ['input_ids', 'attention_mask', 'labels'],
     num_rows: 500
 }))

In [10]:
model = AutoModelForSeq2SeqLM.from_pretrained(hf_model_name).to(device)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

out_dir = "./models/week9_t5small_opusbooks_de_en"
os.makedirs(out_dir, exist_ok=True)

# Note: overwrite_output_dir is removed in transformers v5 as it is the default behavior
training_args = TrainingArguments(
    output_dir=out_dir,
    max_steps=300,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-4,
    weight_decay=0.01,
    warmup_steps=30,
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    report_to="none",
    seed=seed,
    fp16=False,  # Set to False on local CPU/MPS runtime
)

# Note: tokenizer parameter is renamed to processing_class in transformers v5 Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
)

trainer

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [ ]:
# CPU-realistic run: keep this small
trainer.train()

# If you already trained, point to a checkpoint folder here:
# ckpt = "./models/week9_t5small_opus100_en_de/checkpoint-300"
# model = AutoModelForSeq2SeqLM.from_pretrained(ckpt).to(device)

print("Ready: uncomment trainer.train() to fine-tune.")

/Users/mindy/Documents/Learning/GenAI/LLM-transformers/AI-maths-foundations/ai-math-env/lib/python3.12/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss


In [ ]:
# Translation demo (works before/after fine-tuning)
from transformers import set_seed

set_seed(seed)
src_de = "Ich habe diesen Film wirklich genossen, aber das Ende war enttäuschend."
inp = tokenizer(prefix + src_de, return_tensors="pt").to(device)

gen = model.generate(
    **inp,
    max_new_tokens=80,
    num_beams=4,
)

print("DE:", src_de)
print("EN predicted:", tokenizer.decode(gen[0], skip_special_tokens=True))